# Breast Cancer Binary Classification Practice - Answer Key

## 주제: 유방암 데이터셋으로 이진분류 모델 만들기

이 파일은 학생용 실습 파일의 정답/해설 버전입니다.

scikit-learn의 `load_breast_cancer()` 데이터셋을 사용하여 PyTorch 이진분류 모델을 학습하고 평가합니다.

---

## 핵심 포인트

이번 문제는 binary classification입니다.

```text
입력 X: 종양의 여러 측정값
정답 y: malignant 또는 benign
```

이 정답 파일에서는 다음 방식을 사용합니다.

- output layer: 1개
- loss function: `BCEWithLogitsLoss`
- 평가 시 sigmoid 적용
- threshold 0.5 기준으로 0/1 예측

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 1. 데이터셋 불러오기

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target names:", data.target_names)

## 2. 데이터프레임으로 확인하기

In [ ]:
df = pd.DataFrame(X, columns=data.feature_names)
df["target"] = y

df.head()

## 3. Target 분포 확인하기

In [ ]:
print("Target names:", data.target_names)
print(df["target"].value_counts())

plt.figure(figsize=(5, 4))
df["target"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], data.target_names, rotation=0)
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Target Distribution")
plt.grid(axis="y")
plt.show()

## 4. Train/Test Split

`stratify=y`를 사용하면 train/test에 class 비율이 비슷하게 유지됩니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 5. Feature Scaling

주의: scaler는 train 데이터에만 `fit`하고, test 데이터에는 `transform`만 적용합니다.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Before scaling:")
print(X_train[:2])

print("\nAfter scaling:")
print(X_train_scaled[:2])

## 6. PyTorch Tensor로 변환하기

`BCEWithLogitsLoss`를 사용할 것이므로 y는 float 타입이어야 합니다.

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

## 7. DataLoader 만들기

In [ ]:
batch_size = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

for batch_X, batch_y in train_loader:
    print("batch_X:", batch_X.shape)
    print("batch_y:", batch_y.shape)
    break

## 8. 이진분류 모델 정의하기

출력 layer의 크기는 1입니다.

`BCEWithLogitsLoss`를 사용할 것이므로 마지막에 `Sigmoid`를 넣지 않습니다.

In [ ]:
class BreastCancerClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train_tensor.shape[1]
model = BreastCancerClassifier(input_dim)

print(model)

## 9. Loss Function과 Optimizer 설정하기

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 10. 모델 학습하기

In [ ]:
epochs = 100
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        logits = model(batch_X)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

## 11. Loss 그래프 확인하기

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()

## 12. Test 데이터로 평가하기

모델 출력은 logit입니다.

따라서 평가할 때는 sigmoid를 적용해서 확률로 바꾼 뒤, 0.5 이상이면 1, 아니면 0으로 예측합니다.

In [ ]:
model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs >= 0.5).float()

y_true = y_test_tensor.numpy().flatten()
y_pred = test_preds.numpy().flatten()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

## 13. Classification Report 확인하기

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=data.target_names
))

## 14. Confusion Matrix 시각화하기

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
)

disp.plot()
plt.title("Confusion Matrix")
plt.show()

## 15. 예측 확률 일부 확인하기

In [ ]:
result_df = pd.DataFrame({
    "Actual": y_true.astype(int),
    "Predicted": y_pred.astype(int),
    "Probability_of_Class_1": test_probs.numpy().flatten()
})

result_df.head(10)

# 추가 비교: sklearn baseline

작은 tabular 데이터에서는 PyTorch MLP뿐 아니라 Logistic Regression이나 RandomForest 같은 전통적인 머신러닝 모델도 잘 작동합니다.

## 16. Logistic Regression baseline

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

log_pred = log_reg.predict(X_test_scaled)

print("Logistic Regression")
print(f"Accuracy : {accuracy_score(y_test, log_pred):.4f}")
print(f"Precision: {precision_score(y_test, log_pred):.4f}")
print(f"Recall   : {recall_score(y_test, log_pred):.4f}")
print(f"F1-score : {f1_score(y_test, log_pred):.4f}")

## 17. RandomForest baseline

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest")
print(f"Accuracy : {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"Recall   : {recall_score(y_test, rf_pred):.4f}")
print(f"F1-score : {f1_score(y_test, rf_pred):.4f}")

## 18. Feature Importance 확인하기

RandomForest를 사용하면 어떤 feature가 분류에 중요했는지 간단히 확인할 수 있습니다.

In [ ]:
importance_df = pd.DataFrame({
    "feature": data.feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(10)

## 19. Feature Importance 시각화하기

In [ ]:
plt.figure(figsize=(8, 6))
top_features = importance_df.head(10)

plt.barh(top_features["feature"][::-1], top_features["importance"][::-1])
plt.xlabel("Importance")
plt.title("Top 10 Feature Importances")
plt.grid(axis="x")
plt.show()

# 정리

이번 실습의 핵심은 다음과 같습니다.

```text
1. Breast Cancer 데이터셋은 이진분류 문제이다.
2. feature scaling이 중요하다.
3. BCEWithLogitsLoss를 쓰면 모델 마지막에 sigmoid를 넣지 않는다.
4. 평가할 때는 sigmoid + threshold 0.5를 사용한다.
5. Accuracy만 보지 말고 Precision, Recall, F1-score, Confusion Matrix도 함께 확인한다.
6. 작은 tabular 데이터에서는 sklearn baseline과 비교하는 것이 좋다.
```